# Knowledge Distillation with PyTorch DDP on Kubeflow Trainer

Knowledge distillation is a model compression technique where a small **student** model is trained to mimic a larger, more accurate **teacher** model. Instead of training the student only on hard labels (one-hot ground truth), the student also learns from the teacher's **soft predictions** -- the full probability distribution over all classes.

**Why it matters for infrastructure:**
- Smaller models require fewer compute resources for inference (cheaper serving, lower latency)
- Enables deployment on edge devices and resource-constrained environments
- Train once with expensive hardware, deploy the lightweight student everywhere

**How it works:**
1. Train a large teacher model to high accuracy
2. Use temperature-scaled softmax to produce soft probability distributions from the teacher
3. Train the student to match both the teacher's soft predictions (via KL divergence) and the true labels (via cross-entropy)

This notebook demonstrates knowledge distillation on CIFAR-10 using PyTorch DDP, running locally and scaling to multiple nodes with Kubeflow TrainJob.

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Install the PyTorch Dependencies

You also need to install PyTorch and Torchvision to be able to run the example locally:

In [ ]:
!pip install torch==2.9.1
!pip install torchvision==0.22.1

## Understanding Knowledge Distillation

### Hard Labels vs Soft Labels

Traditional training uses **hard labels** (one-hot vectors like `[0, 0, 1, ..., 0]`). These contain no information about relationships between classes. A teacher model's **soft labels** (e.g., `[0.01, 0.05, 0.80, ..., 0.02]`) reveal richer structure -- for example, that a cat image has some similarity to a dog but very little to a truck.

### Temperature Parameter (T)

The temperature `T` controls how soft the probability distribution is:
- `T = 1`: Standard softmax (peaky distribution)
- `T > 1`: Softer distribution that reveals more about class relationships
- Higher T produces more uniform distributions, exposing the "dark knowledge" in the teacher

### Combined Loss Function

The distillation loss combines two terms:

```
loss = alpha * KL_div(student_soft, teacher_soft) * T^2 + (1 - alpha) * CE(student_hard, true_labels)
```

- **Soft target loss**: KL divergence between temperature-scaled student and teacher outputs, multiplied by `T^2` to compensate for the gradient magnitude reduction from temperature scaling
- **Hard target loss**: Standard cross-entropy between student predictions and ground truth labels
- **alpha**: Balances the two losses (0.5 is a standard starting point)

## Define the Training Function

The training function performs three phases:
1. **Phase 1**: Train the teacher model (deeper CNN) on CIFAR-10
2. **Phase 2**: Distill the teacher's knowledge into a student model (lighter CNN)
3. **Phase 3**: Train a baseline student without distillation for comparison

All three phases use PyTorch Distributed Data Parallel (DDP).

In [ ]:
def train_knowledge_distillation():
    import os

    import torch
    import torch.distributed as dist
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, transforms

    # -- Model Definitions --------------------------------------------------

    class TeacherModel(nn.Module):
        """Deeper CNN with ~1.2M parameters."""

        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(3, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(128, 256, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)),
            )
            self.classifier = nn.Linear(256, 10)

        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)
            x = self.classifier(x)
            return x

    class StudentModel(nn.Module):
        """Lighter CNN with ~268K parameters."""

        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(3, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((4, 4)),
            )
            self.classifier = nn.Sequential(
                nn.Linear(32 * 4 * 4, 256),
                nn.ReLU(),
                nn.Linear(256, 10),
            )

        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)
            x = self.classifier(x)
            return x

    # -- Helper: Evaluate accuracy on the test set -------------------------

    def evaluate(model, test_loader, device):
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return 100.0 * correct / total

    # -- Distributed setup --------------------------------------------------

    device_type, backend = (
        ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    )
    print(f"Using Device: {device_type}, Backend: {backend}")

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    device = torch.device(f"{device_type}:{local_rank}")

    # -- Dataset ------------------------------------------------------------

    transform_train = transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            ),
        ]
    )
    transform_test = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
            ),
        ]
    )

    # Download dataset only on local_rank 0 to avoid race conditions.
    if local_rank == 0:
        datasets.CIFAR10("./data", train=True, download=True)
        datasets.CIFAR10("./data", train=False, download=True)
    dist.barrier()

    train_dataset = datasets.CIFAR10(
        "./data", train=True, download=False, transform=transform_train
    )
    test_dataset = datasets.CIFAR10(
        "./data", train=False, download=False, transform=transform_test
    )

    train_sampler = DistributedSampler(train_dataset)
    train_loader = DataLoader(
        train_dataset, batch_size=128, sampler=train_sampler, num_workers=2
    )
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

    # -- Hyperparameters ----------------------------------------------------

    temperature = 4.0
    alpha = 0.5
    num_epochs_teacher = 5
    num_epochs_student = 5

    # ==================================================================
    # Phase 1: Train the Teacher Model
    # ==================================================================
    if dist.get_rank() == 0:
        print("\n" + "=" * 60)
        print("Phase 1: Training Teacher Model")
        print("=" * 60)

    teacher = nn.parallel.DistributedDataParallel(TeacherModel().to(device))
    optimizer = torch.optim.SGD(teacher.parameters(), lr=0.1, momentum=0.9)

    for epoch in range(1, num_epochs_teacher + 1):
        teacher.train()
        train_sampler.set_epoch(epoch)
        running_loss = 0.0
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = teacher(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        if dist.get_rank() == 0:
            acc = evaluate(teacher, test_loader, device)
            print(
                f"Teacher Epoch {epoch}/{num_epochs_teacher} - "
                f"Loss: {running_loss / len(train_loader):.4f}, "
                f"Test Accuracy: {acc:.2f}%"
            )

    teacher.eval()
    if dist.get_rank() == 0:
        teacher_acc = evaluate(teacher, test_loader, device)
        print(f"\nFinal Teacher Accuracy: {teacher_acc:.2f}%")

    # ==================================================================
    # Phase 2: Distill Teacher Knowledge into Student
    # ==================================================================
    if dist.get_rank() == 0:
        print("\n" + "=" * 60)
        print("Phase 2: Knowledge Distillation (Teacher -> Student)")
        print(f"Temperature: {temperature}, Alpha: {alpha}")
        print("=" * 60)

    student_kd = nn.parallel.DistributedDataParallel(StudentModel().to(device))
    optimizer = torch.optim.SGD(student_kd.parameters(), lr=0.1, momentum=0.9)

    for epoch in range(1, num_epochs_student + 1):
        student_kd.train()
        train_sampler.set_epoch(epoch + num_epochs_teacher)  # Different shuffle seed
        running_loss = 0.0
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            # Get teacher predictions (frozen)
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            # Get student predictions
            student_logits = student_kd(inputs)

            # Soft target loss: KL divergence with temperature scaling
            soft_loss = F.kl_div(
                F.log_softmax(student_logits / temperature, dim=1),
                F.softmax(teacher_logits / temperature, dim=1),
                reduction="batchmean",
            ) * (temperature ** 2)

            # Hard target loss: standard cross-entropy
            hard_loss = F.cross_entropy(student_logits, labels)

            # Combined distillation loss
            loss = alpha * soft_loss + (1 - alpha) * hard_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        if dist.get_rank() == 0:
            acc = evaluate(student_kd, test_loader, device)
            print(
                f"Distilled Student Epoch {epoch}/{num_epochs_student} - "
                f"Loss: {running_loss / len(train_loader):.4f}, "
                f"Test Accuracy: {acc:.2f}%"
            )

    if dist.get_rank() == 0:
        distilled_acc = evaluate(student_kd, test_loader, device)
        print(f"\nFinal Distilled Student Accuracy: {distilled_acc:.2f}%")

    # ==================================================================
    # Phase 3: Train Baseline Student (without distillation)
    # ==================================================================
    if dist.get_rank() == 0:
        print("\n" + "=" * 60)
        print("Phase 3: Training Baseline Student (no distillation)")
        print("=" * 60)

    student_baseline = nn.parallel.DistributedDataParallel(StudentModel().to(device))
    optimizer = torch.optim.SGD(student_baseline.parameters(), lr=0.1, momentum=0.9)

    for epoch in range(1, num_epochs_student + 1):
        student_baseline.train()
        train_sampler.set_epoch(epoch + num_epochs_teacher + num_epochs_student)
        running_loss = 0.0
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = student_baseline(inputs)
            loss = F.cross_entropy(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        if dist.get_rank() == 0:
            acc = evaluate(student_baseline, test_loader, device)
            print(
                f"Baseline Student Epoch {epoch}/{num_epochs_student} - "
                f"Loss: {running_loss / len(train_loader):.4f}, "
                f"Test Accuracy: {acc:.2f}%"
            )

    # ==================================================================
    # Final Comparison
    # ==================================================================
    if dist.get_rank() == 0:
        teacher_final = evaluate(teacher, test_loader, device)
        distilled_final = evaluate(student_kd, test_loader, device)
        baseline_final = evaluate(student_baseline, test_loader, device)

        print("\n" + "=" * 60)
        print("RESULTS SUMMARY")
        print("=" * 60)
        print(f"Teacher Model (~1.2M params):           {teacher_final:.2f}%")
        print(f"Distilled Student (~268K params):       {distilled_final:.2f}%")
        print(f"Baseline Student (~268K params):        {baseline_final:.2f}%")
        print(f"Distillation Improvement:               {distilled_final - baseline_final:+.2f}%")
        print("=" * 60)

    # Cleanup
    dist.barrier()
    if dist.get_rank() == 0:
        print("\nTraining is finished")
    dist.destroy_process_group()

## Run the Training Locally

We can submit the training function to the local Trainer client to run it in an isolated subprocess.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# List available runtimes
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

# Submit training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_knowledge_distillation,
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

## Scale PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes.

`TrainerClient()` verifies that you have required access to the Kubernetes cluster.

Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

Additionally, it might show available accelerator type and number of available resources.

In [ ]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above models on 2 PyTorch nodes. The training function runs all three phases (teacher training, knowledge distillation, baseline student) in sequence.

In [ ]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_knowledge_distillation,
        # Set how many PyTorch nodes you want to use for distributed training.
        num_nodes=2,
        # Set the resources for each PyTorch node.
        resources_per_node={
            "cpu": 4,
            "memory": "8Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes.
            # "nvidia.com/gpu": 1,
        },
    ),
    runtime=torch_runtime,
)

## Check the TrainJob Steps

You can check the components of TrainJob that's created.

Since the TrainJob performs distributed training across 2 nodes, it generates 2 steps: `trainer-node-0` and `trainer-node-1`.

You can get the individual status for each of these steps.

In [ ]:
# Wait for the running status.
client.wait_for_job_status(name=job_name, status={"Running"})

In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

## Watch the TrainJob Logs

We can use the `get_job_logs()` API to get the TrainJob logs.

The logs will show all three training phases: teacher training, knowledge distillation, and baseline student training, followed by a results summary comparing all three models.

In [ ]:
for logline in client.get_job_logs(job_name, follow=True):
    print(logline)

## Delete the TrainJob

When TrainJob is finished, you can delete the resource.

In [ ]:
# client.delete_job(job_name)